In [ ]:
import os
import scanpy as sc
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import random
from LingerGRN.pseudo_bulk import *
from LingerGRN.preprocess import *
import LingerGRN.LINGER_tr as LINGER_tr
import LingerGRN.LL_net as LL_net
from LingerGRN.TF_activity import *

sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 5), facecolor='white')
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

os.chdir(path_to_wd)

## Prepare the input data

In [ ]:
adata_full = sc.read("Reproducibility/Data/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

In [ ]:
pre_BCG = ["BC_011","BC_016","BC_023","BC_033","BC_037"]
post_BCG = ['BC_027','BC_032',"BC_039","BC_040",'BC_043',"BC_044","BC_047","BC_048"]
BCG = pre_BCG + post_BCG
adata_tmp = adata_full[adata_full.obs['sample'].isin(BCG)].copy()

In [ ]:
lineage_list = ['CD4_T', 'CD8_T_NK_ILC', 'Myeloid']
lineage = 'CD4_T' ## Choose one from lineage_list

adata = adata_tmp[adata_tmp.obs["lineage"] == lineage].copy()

In [ ]:
dir_path = f'Reproducibility/Results/LINGER/BCG/{lineage}/'
os.makedirs(dir_path, exist_ok = True)

outdir=f"{dir_path}output/"
os.makedirs(outdir, exist_ok = True)

datadir=f"{dir_path}data/"
os.makedirs(datadir, exist_ok = True)

adata.obs["label"] = adata.obs["celltype"]
adata.obs['barcode'] = adata.obs_names

label = pd.DataFrame({'barcode_use': adata.obs_names,
                      'label': adata.obs["label"]
                     })

In [ ]:
adata_RNA  = adata[:, adata.var.modality == "Gene Expression"].copy()
adata_ATAC = adata[:, adata.var.modality == "Peaks"].copy()

adata_RNA = adata_RNA[:, adata_RNA.var['black_list']=='keep']
sc.pp.filter_genes(adata_RNA, min_cells=3)
sc.pp.filter_genes(adata_ATAC, min_cells=3)

adata_RNA.var['gene_ids'] = adata_RNA.var_names
adata_ATAC.var['gene_ids'] = adata_ATAC.var_names

adata_RNA.raw = adata_RNA
adata_ATAC.raw = adata_ATAC

## Generate the pseudo-bulk/metacell 

In [ ]:
from LingerGRN.pseudo_bulk import *
samplelist=list(set(adata_ATAC.obs['sample'].values)) # sample is generated from cell barcode 
tempsample=samplelist[0]
TG_pseudobulk=pd.DataFrame([])
RE_pseudobulk=pd.DataFrame([])
singlepseudobulk = (adata_RNA.obs['sample'].unique().shape[0]*adata_RNA.obs['sample'].unique().shape[0]>100)
for tempsample in samplelist:
    adata_RNAtemp=adata_RNA[adata_RNA.obs['sample']==tempsample]
    adata_ATACtemp=adata_ATAC[adata_ATAC.obs['sample']==tempsample]
    TG_pseudobulk_temp,RE_pseudobulk_temp=pseudo_bulk(adata_RNAtemp,adata_ATACtemp,singlepseudobulk)                
    TG_pseudobulk=pd.concat([TG_pseudobulk, TG_pseudobulk_temp], axis=1)
    RE_pseudobulk=pd.concat([RE_pseudobulk, RE_pseudobulk_temp], axis=1)
    RE_pseudobulk[RE_pseudobulk > 100] = 100

adata_ATAC.write(f'{datadir}adata_ATAC.h5ad')
adata_RNA.write(f'{datadir}adata_RNA.h5ad')
TG_pseudobulk=TG_pseudobulk.fillna(0)
RE_pseudobulk=RE_pseudobulk.fillna(0)

pd.DataFrame(adata_ATAC.var['gene_ids']).to_csv(f'{datadir}Peaks.txt',header=None,index=None)
TG_pseudobulk.to_csv(f'{datadir}TG_pseudobulk.tsv')
RE_pseudobulk.to_csv(f'{datadir}RE_pseudobulk.tsv')

In [ ]:
adata_ATAC.var['gene_ids'].to_csv(f"{datadir}Peaks.txt",header=None,index=None)

## Overlap the region with general GRN

In [ ]:
os.chdir(dir_path)
method = 'LINGER'
Datadir='/home/k-makino/reference/LINGER/' 
GRNdir='/home/k-makino/reference/LINGER/data_bulk/'

preprocess(TG_pseudobulk = TG_pseudobulk, RE_pseudobulk = RE_pseudobulk,
           GRNdir = GRNdir, outdir = outdir, 
           genome = 'hg38', method = 'LINGER')

## Train for the LINGER model

In [ ]:
activef='ReLU'
LINGER_tr.training(GRNdir,method,outdir,activef)   

## Cell population gene regulatory network

In [ ]:
genome = 'hg38'
network = 'cell population'

# TF binding potential
LL_net.TF_RE_binding(GRNdir,adata_RNA_filt,adata_ATAC_filt,genome,method,outdir)
# cis-regulatory network
LL_net.cis_reg(GRNdir,adata_RNA_filt,adata_ATAC_filt,genome,method,outdir)
# trans-regulatory network
LL_net.trans_reg(GRNdir,method,outdir,genome)

TF_activity = regulon(outdir,adata_RNA,GRNdir,network,genome)
TF_activity = TF_activity.dropna(how='all').dropna(how='all', axis=1)
TF_activity.to_csv(f"Reproducibility/Results/LINGER/BCG/cell_population_TF_activity_{lineage}_BCG.txt", sep='\t')

### Scaled RNA/TF activity matrix (paired)

In [ ]:
pre_BCG = ["BC_011","BC_023","BC_033","BC_037"]
post_BCG = ["BC_039","BC_044","BC_048","BC_047"]
BCG = pre_BCG + post_BCG

In [ ]:
dict = {'CD4_T': 'CD4_Tconv', 'Myeloid': 'Mono_Mac'}

for lineage in ['CD4_T', 'Myeloid']:
    subset = dict[lineage]
    adata = sc.read(f'Reproducibility/Data/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad')
    adata_RNA  = adata[:, adata.var.modality == "Gene Expression"].copy()
    adata_RNA = adata_RNA[adata_RNA.obs['sample'].isin(BCG)]
    adata_RNA = adata_RNA[adata_RNA.obs['coarse_celltype'].isin([dict[lineage]])]

    sc.pp.scale(adata_RNA, max_value=10)

    TF_activity = pd.read_csv(f"Reproducibility/Results/LINGER/BCG/cell_population_TF_activity_{lineage}_BCG.txt",sep='\t',index_col=0)
    adata_TF = sc.AnnData(TF_activity.T.loc[adata_RNA.obs_names,:].copy(), obs=adata_RNA.obs)
    sc.pp.scale(adata_TF, max_value=10)

    TF_use = np.intersect1d(adata_RNA.var_names, TF_activity.index)
    exp_df = pd.DataFrame(adata_RNA[:,TF_use].X, index = adata_RNA.obs_names, columns = TF_use)
    TF_df = pd.DataFrame(adata_TF[:,TF_use].X, index = adata_TF.obs_names, columns = TF_use)

    ###################### 
    pre = adata_RNA[adata_RNA.obs['sample'].isin(pre_BCG)]
    post = adata_RNA[adata_RNA.obs['sample'].isin(post_BCG)]

    ## average
    pre_exp_df = exp_df.loc[pre.obs_names,:].copy()
    post_exp_df = exp_df.loc[post.obs_names,:].copy()

    pre_TF_df = TF_df.loc[pre.obs_names,:].copy()
    post_TF_df = TF_df.loc[post.obs_names,:].copy()

    exp_diff = post_exp_df.mean(axis='index') - pre_exp_df.mean(axis='index')
    TF_diff = post_TF_df.mean(axis='index') - pre_TF_df.mean(axis='index')
    
    exp_TF_df = pd.DataFrame({'EXP': exp_diff, 'TF': TF_diff})
    exp_TF_df.to_csv(f"Reproducibility/Results/LINGER/BCG/cell_population_exp_TF_activity_zscore_{subset}_BCG_paired.txt", sep='\t')